# PathoVision - complete re-execution

Runs every stage of `reanalysis/` on a Colab GPU (L4 recommended).

* checkpoints, predictions and results are stored on **Google Drive** (`REANALYSIS_ROOT`), so they survive session resets;
* images are kept on the **local disk** (`REANALYSIS_DATA_ROOT`) and archived once to Drive, so later sessions restore them in a minute instead of re-downloading;
* every stage is resumable: just re-run the cell after a session limit.

Run the cells in order. Only the Kaggle cell needs your input.

In [ ]:
#@title 1. Mount Google Drive and set the folders
from google.colab import drive
drive.mount('/content/drive')
import os
ROOT = '/content/drive/MyDrive/pathovision_reanalysis'   # runs/ and results/ (persistent)
DATA = '/content/pv_data'                                 # datasets/ (fast local disk)
REPO = '/content/pathovision_model'
os.makedirs(ROOT, exist_ok=True); os.makedirs(DATA, exist_ok=True)
os.environ['REANALYSIS_ROOT'] = ROOT
os.environ['REANALYSIS_DATA_ROOT'] = DATA
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
#@title 2. Clone the repository and install dependencies
GITHUB_REPO = 'https://github.com/alexroehrs/pathovision_model.git'  #@param {type:"string"}
import os
if not os.path.exists(REPO):
    !git clone --depth 1 {GITHUB_REPO} {REPO}
else:
    !cd {REPO} && git pull --ff-only
%cd {REPO}
!pip install -q -r requirements.txt statsmodels 2>&1 | tail -n 2
os.environ['PYTHONPATH'] = REPO
import torch; print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

### 3. Kaggle credentials (your own)

The datasets are downloaded with the Kaggle CLI. On kaggle.com go to *Settings > API > Create New Token* and copy the token it shows (a long string starting with `KGAT_`). Paste it into the hidden prompt of the next cell. It is kept only in this Colab session (`KAGGLE_API_TOKEN` and `~/.kaggle/access_token`) and is never printed or copied anywhere else.

If you prefer the old-style `kaggle.json` (*Create Legacy API Key*), upload it to `~/.kaggle/kaggle.json` instead; the scripts accept either.

In [ ]:
#@title 3. Paste your Kaggle API token (skipped if the datasets archive already exists on Drive)
import os, pathlib
archive = pathlib.Path(ROOT) / 'datasets.tar'
if archive.exists():
    print('datasets archive found on Drive; Kaggle credentials not needed')
else:
    !pip install -q -U kaggle 2>&1 | tail -n 1
    from getpass import getpass
    token = getpass('Kaggle API token (input is hidden): ').strip()
    assert token, 'empty token'
    os.environ['KAGGLE_API_TOKEN'] = token
    kdir = pathlib.Path.home() / '.kaggle'; kdir.mkdir(exist_ok=True)
    (kdir / 'access_token').write_text(token); os.chmod(kdir / 'access_token', 0o600)
    del token
    !kaggle datasets list -s dermnet 2>&1 | head -n 3   # quick authentication check


In [ ]:
#@title 4. Build (or restore) the internal and external datasets
import pathlib, subprocess, os
archive = pathlib.Path(ROOT) / 'datasets.tar'
if archive.exists() and not (pathlib.Path(DATA) / 'data' / 'test').exists():
    print('restoring datasets from Drive ...')
    !tar -xf "{archive}" -C "{DATA}"
if not (pathlib.Path(DATA) / 'data' / '.reanalysis_done.json').exists():
    !python reanalysis/s01_build_internal.py
if not (pathlib.Path(DATA) / 'external_unified' / '.reanalysis_done').exists():
    !python reanalysis/s02_build_external.py
if not archive.exists():
    print('archiving datasets to Drive for the next sessions ...')
    !tar -cf "{archive}" -C "{DATA}" data external_unified
!ls "{DATA}/data" && cat "{ROOT}/results/dataset_qc/split_counts.csv"

### 5. Training - 9 configurations x 3 seeds

This is the long part (several GPU-hours per seed on an L4). The cell is resumable: re-run it after any interruption and it continues with the first run whose `done.json` is missing. To train one seed at a time, change `SEEDS`.

In [ ]:
#@title 5. Train all configurations
SEEDS = "42 43 44"  #@param {type:"string"}
!python reanalysis/s03_train_all.py --seeds {SEEDS}

In [ ]:
#@title 6. Predictions, ensembles/hybrid, cross-validation, Grad-CAM (GPU)
!python reanalysis/s04_predict_all.py
!python reanalysis/s05_ensemble_hybrid.py
!python reanalysis/s06_cross_validate_full.py
!python reanalysis/s12_gradcam_quant.py

In [ ]:
#@title 7. Tables, statistics and figures (CPU)
!python reanalysis/run_all.py --stages 07 08 09 10 11 13 14

In [ ]:
#@title 8. Bundle the results (tables, figures, predictions - no checkpoints) for download
import pathlib, shutil, os
bundle = pathlib.Path(ROOT) / 'results_bundle'
if bundle.exists(): shutil.rmtree(bundle)
shutil.copytree(pathlib.Path(ROOT) / 'results', bundle / 'results')
for p in pathlib.Path(ROOT, 'runs').rglob('*'):
    if p.is_file() and (p.suffix in ('.csv', '.json', '.yaml') or p.name.startswith('emb_')):
        dst = bundle / 'runs' / p.relative_to(pathlib.Path(ROOT, 'runs')); dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(p, dst)
zip_path = shutil.make_archive(str(pathlib.Path(ROOT) / 'results_bundle'), 'zip', bundle)
print('bundle written:', zip_path, f'({os.path.getsize(zip_path)/1e6:.1f} MB)')
from google.colab import files; files.download(zip_path)